In [ ]:
STARTING_DATE = "2025-03-20"
END_DATE = "2025-09-30"

MARGINALPDBC = "marginalpdbc"
CAPACIDAD_INTER_PBC = "capacidad_inter_pbc"

In [27]:
SELECTED_FILE = MARGINALPDBC

In [28]:
OMIE_DOWNLOAD_URL = (
    "https://www.omie.es/es/file-download?parents="
    + SELECTED_FILE
    + "&filename="
    + SELECTED_FILE
    + "_{date}.{version}"
)
SAVING_PATH = "data/" + SELECTED_FILE + "/" + SELECTED_FILE + "_{date}.{version}"
import requests
import pandas as pd


def get_omie_download_url(date, version):
    """
    Returns the OMIE download URL for a given date and version.

    :param date: The date in the format YYYYMMDD.
    :param version: The version of the file.
    :return: The complete download URL.
    """
    date_formatted = pd.to_datetime(date).strftime("%Y%m%d")
    return OMIE_DOWNLOAD_URL.format(date=date_formatted, version=version)


def download_omie_file(date, version):
    """
    Downloads the OMIE file for a given date and version.

    :param date: The date in the format YYYYMMDD.
    :param version: The version of the file.
    :return: The content of the downloaded file.
    """

    url = get_omie_download_url(date, version)
    # max 10 seconds timeout
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            return response.content
        else:
            return None
    except requests.RequestException as e:
        return "timeout"


dates = pd.date_range(start=STARTING_DATE, end=END_DATE, freq="D")
timeout_dates = []

for date in dates:
    ended_attempts = False
    version = 1
    while not ended_attempts:
        content = download_omie_file(date.strftime("%Y-%m-%d"), version)
        if content:
            file_path = SAVING_PATH.format(
                date=date.strftime("%Y%m%d"), version=version
            )

            with open(file_path, "wb") as file:
                file.write(content)

            print(f"Downloaded and saved: {file_path}")
            ended_attempts = True
        elif content == "timeout":
            timeout_dates.append(date.strftime("%Y-%m-%d"))
            ended_attempts = True
        else:
            print(
                f"Failed to download {date.strftime('%Y-%m-%d')} version {version}. Retrying..."
            )
            version += 1
            if version > 5:
                raise Exception(
                    f"Failed to download {date.strftime('%Y-%m-%d')} after multiple attempts."
                )

print("Download process completed.")
display(f"Timeout dates: {timeout_dates}")

Downloaded and saved: data/marginalpdbc/marginalpdbc_20250320.1
Downloaded and saved: data/marginalpdbc/marginalpdbc_20250321.1
Downloaded and saved: data/marginalpdbc/marginalpdbc_20250322.1
Download process completed.


'Timeout dates: []'